# 4. Проверка гипотезы с Graph-EVT

**Цель:** перевести содержательную гипотезу о связи экстремумов с локальной волатильностью в воспроизводимую численную проверку. Установленная версия `graph_evt_agent` не содержит LLM API: в ней используются публичные классы `EVTConfig`, `GraphConfig`, `InputConfig` и `GraphEVTPipeline`.


In [ ]:
from pathlib import Path
from importlib.metadata import version
import numpy as np
import pandas as pd
from graph_evt_agent import EVTConfig, GraphConfig, GraphEVTPipeline, InputConfig

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/fractal_extreme_series.csv").exists())
DATA = ROOT / "data/fractal_extreme_series.csv"
SEED = 42
df = pd.read_csv(DATA)
x = df["value"].to_numpy()
assert np.isfinite(x).all() and x.ndim == 1


In [ ]:
BASELINE_SIZE = int(0.75 * len(x))
config = EVTConfig(
    baseline_size=BASELINE_SIZE,
    tail_quantile=0.95,
    alarm_probability=0.999,
    persistence=3,
)
pipeline = GraphEVTPipeline(
    config,
    graph=GraphConfig(random_state=SEED),
    input_config=InputConfig(missing="error"),
)
result = pipeline.run_detailed(x)
score = result.detection.indicator

# Проверяемая гипотеза: EVT-кандидаты возникают при повышенной предшествующей
# волатильности. Порог кандидатов обучается только на baseline.
window = 32
returns = pd.Series(x).diff()
lagged_volatility = returns.rolling(window).std().shift(1).to_numpy()
extreme_cutoff = float(np.quantile(score[:BASELINE_SIZE], 0.99))
is_candidate = score > extreme_cutoff
valid = np.isfinite(lagged_volatility)
observed_difference = (
    lagged_volatility[valid & is_candidate].mean()
    - lagged_volatility[valid & ~is_candidate].mean()
)

# Circular block randomisation preserves short-range dependence while breaking
# the alignment between the EVT indicator and volatility.
rng = np.random.default_rng(SEED)
block_size = 64
permuted = []
for _ in range(999):
    shift = int(rng.integers(block_size, len(x) - block_size))
    shifted_candidate = np.roll(is_candidate, shift)
    permuted.append(
        lagged_volatility[valid & shifted_candidate].mean()
        - lagged_volatility[valid & ~shifted_candidate].mean()
    )
p_value = (1 + np.count_nonzero(np.abs(permuted) >= abs(observed_difference))) / (len(permuted) + 1)

test_result = pd.DataFrame([{
    "hypothesis": "EVT candidates have higher preceding 32-step volatility",
    "candidate_threshold": extreme_cutoff,
    "candidate_count": int(is_candidate.sum()),
    "observed_volatility_difference": observed_difference,
    "randomisations": len(permuted),
    "block_size": block_size,
    "two_sided_p_value": p_value,
    "reject_at_0.05": bool(p_value < 0.05),
}])
test_result


In [ ]:
artifact_dir = ROOT / "results"
artifact_dir.mkdir(exist_ok=True)
test_result.to_csv(artifact_dir / "04_hypothesis_test.csv", index=False)
pd.DataFrame({
    "time": df["time"],
    "value": x,
    "evt_indicator": score,
    "lagged_volatility": lagged_volatility,
    "is_candidate": is_candidate,
}).to_csv(artifact_dir / "04_hypothesis_series.csv", index=False)


Результат является тестом ассоциации, а не причинности. Circular block randomisation — учебная диагностическая процедура: вывод зависит от окна, размера блока и порога кандидатов, поэтому эти параметры сохранены в таблице.

**Самопроверка:** что делает гипотезу опровержимой? Почему волатильность сдвинута на один шаг? Что сохраняет блочная рандомизация? Где возникает множественная проверка при переборе параметров?
